# ML v1.2 - feature-group ablation, sparse identity и segment analysis

Этот notebook проверяет, какие группы признаков реально дают качество лучшей CatBoost-модели из `v1.1`. Мы используем тот же датасет `client_profile_v1_0_shuffled.csv` и тот же split, но обучаем несколько вариантов модели: полный набор, полный набор со sparse identity, а также версии без отдельных групп признаков. В конце считаем top-k, пороги и качество на важных сегментах клиентов.


In [11]:
from pathlib import Path
import sys
import subprocess
import importlib.util
import warnings

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Google Drive mount skipped:", exc)

BASE_PATH = Path("/content/drive/MyDrive/ieee-db")
DATA_FILE = BASE_PATH / "client_profile_v1_0_shuffled.csv"
RESULTS_DIR = BASE_PATH / "ML" / "1.2" / "results_ablation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Файл данных:", DATA_FILE)
print("Папка результатов:", RESULTS_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Файл данных: /content/drive/MyDrive/ieee-db/client_profile_v1_0_shuffled.csv
Папка результатов: /content/drive/MyDrive/ieee-db/ML/1.2/results_ablation


In [12]:
if importlib.util.find_spec("catboost") is None:
    print("CatBoost не найден. Устанавливаю catboost...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost"])
else:
    print("CatBoost уже установлен.")

import time
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 200)


CatBoost уже установлен.


In [13]:
df = pd.read_csv(DATA_FILE)
print("Размер таблицы:", df.shape)
print("Распределение target:")
print(df["profile_fraud_label"].value_counts(dropna=False))
print("Доля fraud:", df["profile_fraud_label"].mean())
display(df.head())


Размер таблицы: (174280, 95)
Распределение target:
profile_fraud_label
0    167785
1      6495
Name: count, dtype: int64
Доля fraud: 0.03726761533165022


,client_id,tx_count_total,tx_amount_sum,tx_amount_mean,tx_amount_median_proxy,tx_amount_std,tx_count_standart_flow,tx_count_high_risk_flow,tx_sum_stanadart_flow_proxy_proxy,tx_sum_high_risk_flow_proxy,tx_freq_per_day,daily_activity_share,avg_inter_tx_seconds,short_turnover_share,amount_repeat_share,odd_amount_share,weighted_amount_repeat_share,cash_out_ratio_proxy,MCC_risk_share_proxy,high_risk_vs_mean,crypto_pattern_score,low_history_flag,history_support_score,productcd_nunique,addr2_nunique,card4_mode,card6_mode,tx_dt_span_days,top_email_domain,profile_fraud_label,identity_present,num_missing_identity,identity_rows,non_null_id_values,device_type_nunique,id_01_mean,id_01_median,id_01_count,id_02_mean,id_02_median,id_02_count,id_03_mean,id_03_median,id_03_count,id_04_mean,id_04_median,id_04_count,id_05_mean,id_05_median,id_05_count,id_06_mean,id_06_median,id_06_count,id_07_mean,id_07_median,id_07_count,id_08_mean,id_08_median,id_08_count,id_09_mean,id_09_median,id_09_count,id_10_mean,id_10_median,id_10_count,id_11_mean,id_11_median,id_11_count,id_12_mode,id_13_mode,id_14_mode,id_15_mode,id_16_mode,id_17_mode,id_18_mode,id_19_mode,id_20_mode,id_21_mode,id_22_mode,id_23_mode,id_24_mode,id_25_mode,id_26_mode,id_27_mode,id_28_mode,id_29_mode,id_30_mode,id_31_mode,id_32_mode,id_33_mode,id_34_mode,id_35_mode,id_36_mode,id_37_mode,id_38_mode
0,12221_170.0_173,1,47.950,47.950000,47.950,0.000000,1,0,47.95,0.000,1.000000,1.000000,NaN,0.0,0.000000,1.0,0.000000,1.0,0.0,0.0,0.000000,1,0.1,1,1,mastercard,debit,0.000000,aol.com,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5347__14,1,31.191,31.191000,31.191,0.000000,0,1,0.00,31.191,1.000000,1.000000,NaN,0.0,0.000000,1.0,0.000000,1.0,1.0,1.0,0.968935,1,0.1,1,0,mastercard,debit,0.000000,gmail.com,0,1,14,1,24,1.0,-10.0,-10.0,1.0,232053.0,232053.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,-2.0,-2.0,1.0,-100.0,-100.0,1.0,NaN,NaN,0.0,NaN,NaN,0.0,0.0,0.0,1.0,-6.0,-6.0,1.0,100.0,100.0,1.0,NotFound,52.0,NaN,Found,Found,166.0,13.0,456.0,256.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Found,Found,NaN,chrome 62.0,NaN,NaN,NaN,F,F,T,T
2,7508_420.0_97,3,117.850,39.283333,36.950,7.133645,3,0,117.85,0.000,0.059385,0.058824,2182363.0,0.0,0.000000,1.0,0.000000,1.0,0.0,0.0,0.000000,1,0.3,1,1,visa,debit,50.517662,outlook.com,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,17759_126.0_12,1,103.950,103.950000,103.950,0.000000,1,0,103.95,0.000,1.000000,1.000000,NaN,0.0,0.000000,1.0,0.000000,1.0,0.0,0.0,0.000000,1,0.1,1,1,mastercard,debit,0.000000,gmail.com,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,16092_143.0_74,3,3426.550,1142.183333,1325.880,259.786317,3,0,3426.55,0.000,3.000000,1.000000,2160.0,1.0,0.666667,1.0,0.133333,1.0,0.0,0.0,0.000000,1,0.3,1,1,mastercard,debit,0.050000,gmail.com,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Настройки эксперимента

По умолчанию используем быстрый/medium CatBoost, а не long-run winner. Цель `v1.2` - быстро сравнить группы признаков, а не выжать максимальную метрику из каждой конфигурации. Если после ablation найдется лучший набор признаков, его можно будет отдельно прогнать в long-run режиме.


In [14]:
TARGET_COL = "profile_fraud_label"
ID_COLS = ["client_id"]
LEAKY_COLS = [c for c in ["tx_count_fraud", "fraud_rate"] if c in df.columns]
HIGH_MISSING_THRESHOLD = 0.95

TASK_TYPE = "GPU"  # Лучше оставить CPU из-за ранее встречавшейся CUDA-ошибки в Colab.
RUN_MODE = "medium"  # "smoke", "medium" или "long_confirm".
VARIANT_LIMIT = None  # Например, 3 для быстрой проверки. None - прогнать все варианты.

if RUN_MODE == "smoke":
    CATBOOST_ITERATIONS = 500
    EARLY_STOPPING_ROUNDS = 50
    CATBOOST_DEPTH = 5
    CATBOOST_LEARNING_RATE = 0.05
    CATBOOST_L2 = 3
elif RUN_MODE == "long_confirm":
    # Использовать только после ablation, желательно на 1-2 лучших вариантах.
    CATBOOST_ITERATIONS = 3500
    EARLY_STOPPING_ROUNDS = 150
    CATBOOST_DEPTH = 7
    CATBOOST_LEARNING_RATE = 0.03
    CATBOOST_L2 = 8
else:
    # Основной режим v1.2: быстрее long-run, достаточно надежно для сравнения групп признаков.
    CATBOOST_ITERATIONS = 1400
    EARLY_STOPPING_ROUNDS = 80
    CATBOOST_DEPTH = 6
    CATBOOST_LEARNING_RATE = 0.04
    CATBOOST_L2 = 3

CATBOOST_PARAMS = dict(
    iterations=CATBOOST_ITERATIONS,
    learning_rate=CATBOOST_LEARNING_RATE,
    depth=CATBOOST_DEPTH,
    l2_leaf_reg=CATBOOST_L2,
    loss_function="Logloss",
    eval_metric="PRAUC",
    auto_class_weights="Balanced",
    random_seed=42,
    task_type=TASK_TYPE,
    thread_count=-1,
    allow_writing_files=False,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    verbose=200,
)

print("RUN_MODE:", RUN_MODE)
print("CatBoost params:")
print(CATBOOST_PARAMS)


RUN_MODE: medium
CatBoost params:
{'iterations': 1400, 'learning_rate': 0.04, 'depth': 6, 'l2_leaf_reg': 3, 'loss_function': 'Logloss', 'eval_metric': 'PRAUC', 'auto_class_weights': 'Balanced', 'random_seed': 42, 'task_type': 'GPU', 'thread_count': -1, 'allow_writing_files': False, 'early_stopping_rounds': 80, 'verbose': 200}


## Группы признаков

Группы строятся автоматически по именам колонок. После построения notebook печатает, какие колонки попали в каждую группу, чтобы можно было проверить логику до запуска обучения.


In [15]:
all_candidate_features = [
    c for c in df.columns
    if c not in [TARGET_COL] + ID_COLS + LEAKY_COLS
]

high_missing_cols = [
    c for c in all_candidate_features
    if df[c].isna().mean() > HIGH_MISSING_THRESHOLD
]
base_features = [c for c in all_candidate_features if c not in high_missing_cols]


def is_identity_col(c):
    low = c.lower()
    return (
        c.startswith("id_")
        or low in {"identity_present", "identity_rows", "num_missing_identity", "non_null_id_values", "device_type_nunique"}
        or low.startswith("device")
        or "device" in low
    )


def is_email_col(c):
    low = c.lower()
    return "email" in low or "domain" in low


def is_amount_col(c):
    low = c.lower()
    return (
        "amount" in low
        or low.startswith("tx_sum")
        or "turnover" in low
        or low in {"high_risk_vs_mean"}
    )


def is_activity_time_col(c):
    low = c.lower()
    return (
        "freq" in low
        or "inter_tx" in low
        or "span" in low
        or "activity" in low
        or "turnover_share" in low
        or "dt_" in low
        or low.endswith("_days")
        or "seconds" in low
    )


def is_flow_risk_col(c):
    low = c.lower()
    return (
        "risk" in low
        or "flow" in low
        or "cash_out" in low
        or "mcc" in low
        or "crypto" in low
        or "productcd" in low
    )

identity_cols = [c for c in all_candidate_features if is_identity_col(c)]
email_cols = [c for c in all_candidate_features if is_email_col(c)]
amount_cols = [c for c in all_candidate_features if is_amount_col(c)]
activity_time_cols = [c for c in all_candidate_features if is_activity_time_col(c)]
flow_risk_cols = [c for c in all_candidate_features if is_flow_risk_col(c)]
sparse_identity_cols = [c for c in high_missing_cols if is_identity_col(c)]

feature_groups = {
    "identity": identity_cols,
    "email_domain": email_cols,
    "amount": amount_cols,
    "activity_time": activity_time_cols,
    "flow_risk": flow_risk_cols,
    "sparse_identity": sparse_identity_cols,
}

print("Всего candidate features:", len(all_candidate_features))
print("Base features:", len(base_features))
print("High missing cols:", high_missing_cols)
print()
for group_name, cols in feature_groups.items():
    print("=" * 80)
    print(group_name, "count=", len(cols))
    print(cols)


Всего candidate features: 93
Base features: 82
High missing cols: ['id_07_mean', 'id_07_median', 'id_08_mean', 'id_08_median', 'id_21_mode', 'id_22_mode', 'id_23_mode', 'id_24_mode', 'id_25_mode', 'id_26_mode', 'id_27_mode']

identity count= 65
['identity_present', 'num_missing_identity', 'identity_rows', 'non_null_id_values', 'device_type_nunique', 'id_01_mean', 'id_01_median', 'id_01_count', 'id_02_mean', 'id_02_median', 'id_02_count', 'id_03_mean', 'id_03_median', 'id_03_count', 'id_04_mean', 'id_04_median', 'id_04_count', 'id_05_mean', 'id_05_median', 'id_05_count', 'id_06_mean', 'id_06_median', 'id_06_count', 'id_07_mean', 'id_07_median', 'id_07_count', 'id_08_mean', 'id_08_median', 'id_08_count', 'id_09_mean', 'id_09_median', 'id_09_count', 'id_10_mean', 'id_10_median', 'id_10_count', 'id_11_mean', 'id_11_median', 'id_11_count', 'id_12_mode', 'id_13_mode', 'id_14_mode', 'id_15_mode', 'id_16_mode', 'id_17_mode', 'id_18_mode', 'id_19_mode', 'id_20_mode', 'id_21_mode', 'id_22_mode',

In [16]:
def remove_cols(features, cols_to_remove):
    remove_set = set(cols_to_remove)
    return [c for c in features if c not in remove_set]

feature_variants = {
    "full_base": base_features,
    "with_sparse_identity": sorted(set(base_features + sparse_identity_cols), key=lambda c: all_candidate_features.index(c)),
    "without_identity": remove_cols(base_features, identity_cols),
    "without_email_domain": remove_cols(base_features, email_cols),
    "without_amount": remove_cols(base_features, amount_cols),
    "without_activity_time": remove_cols(base_features, activity_time_cols),
    "without_flow_risk": remove_cols(base_features, flow_risk_cols),
}

if VARIANT_LIMIT is not None:
    feature_variants = dict(list(feature_variants.items())[:VARIANT_LIMIT])

сводка_rows = []
for name, features in feature_variants.items():
    removed = sorted(set(base_features) - set(features))
    added = sorted(set(features) - set(base_features))
    сводка_rows.append({
        "variant": name,
        "n_features": len(features),
        "n_removed_vs_base": len(removed),
        "n_added_vs_base": len(added),
        "removed_cols": ", ".join(removed),
        "added_cols": ", ".join(added),
    })

feature_group_сводка_df = pd.DataFrame(сводка_rows)
feature_group_сводка_df.to_csv(RESULTS_DIR / "ml_v1_2_feature_group_сводка.csv", index=False)
display(feature_group_сводка_df)
print("Feature group сводка сохранен в:", RESULTS_DIR / "ml_v1_2_feature_group_сводка.csv")


,variant,n_features,n_removed_vs_base,n_added_vs_base,removed_cols,added_cols
0,full_base,82,0,0,,
1,with_sparse_identity,93,0,11,,"id_07_mean, id_07_median, id_08_mean, id_08_me..."
2,without_identity,28,54,0,"device_type_nunique, id_01_count, id_01_mean, ...",
3,without_email_domain,81,1,0,top_email_domain,
4,without_amount,71,11,0,"amount_repeat_share, high_risk_vs_mean, odd_am...",
5,without_activity_time,77,5,0,"avg_inter_tx_seconds, daily_activity_share, sh...",
6,without_flow_risk,73,9,0,"MCC_risk_share_proxy, cash_out_ratio_proxy, cr...",


Feature group сводка сохранен в: /content/drive/MyDrive/ieee-db/ML/1.2/results_ablation/ml_v1_2_feature_group_сводка.csv


## Split

Используем одинаковый split для всех вариантов, чтобы сравнение было честным. Доли примерно 70% train, 15% validation, 15% test.


In [17]:
y = df[TARGET_COL].astype(int)
all_idx = np.arange(len(df))

train_val_idx, test_idx = train_test_split(
    all_idx,
    test_size=0.15,
    random_state=42,
    stratify=y,
)

train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.1764705882,
    random_state=42,
    stratify=y.iloc[train_val_idx],
)

print("Train:", len(train_idx), "fraud rate:", y.iloc[train_idx].mean())
print("Val:  ", len(val_idx), "fraud rate:", y.iloc[val_idx].mean())
print("Test: ", len(test_idx), "fraud rate:", y.iloc[test_idx].mean())


Train: 121996 fraud rate: 0.037271713826682845
Val:   26142 fraud rate: 0.037258052176574095
Test:  26142 fraud rate: 0.037258052176574095


## Метрики

Сохраняем общие метрики, top-k, пороги и сегментный анализ. Для выбора модели используем validation PR-AUC, а test остается подтверждением результата.


In [18]:
def safe_roc_auc(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_prob)


def safe_pr_auc(y_true, y_prob):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return average_precision_score(y_true, y_prob)


def calc_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "roc_auc": safe_roc_auc(y_true, y_prob),
        "pr_auc": safe_pr_auc(y_true, y_prob),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "pred_positive_rate": y_pred.mean(),
        "pred_positive_count": int(y_pred.sum()),
        "support": int(len(y_true)),
        "positive_support": int(y_true.sum()),
    }


def build_topk_table(y_true, y_prob, rates=(0.01, 0.03, 0.05, 0.10, 0.15)):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    order = np.argsort(-y_prob)
    total_positive = y_true.sum()
    rows = []
    for rate in rates:
        k = max(1, int(np.ceil(len(y_true) * rate)))
        selected = order[:k]
        positives_found = int(y_true[selected].sum())
        rows.append({
            "top_rate": rate,
            "top_k": k,
            "precision_at_k": positives_found / k,
            "recall_at_k": positives_found / total_positive if total_positive else 0.0,
            "fraud_found": positives_found,
            "total_fraud": int(total_positive),
        })
    return pd.DataFrame(rows)


def build_threshold_table(y_true, y_prob, thresholds=(0.50, 0.70, 0.75, 0.80, 0.85, 0.90)):
    rows = []
    for threshold in thresholds:
        row = calc_metrics(y_true, y_prob, threshold=threshold)
        row["threshold"] = threshold
        rows.append(row)
    return pd.DataFrame(rows)


def prepare_catboost_frame(frame, categorical_cols):
    out = frame.copy()
    for c in categorical_cols:
        out[c] = out[c].astype("object").where(out[c].notna(), "MISSING").astype(str)
    return out


def make_pool(features, idx):
    X_part = df.iloc[idx][features].copy()
    y_part = y.iloc[idx].copy()
    categorical_cols = X_part.select_dtypes(include=["object", "category"]).columns.tolist()
    X_part = prepare_catboost_frame(X_part, categorical_cols)
    cat_feature_indices = [X_part.columns.get_loc(c) for c in categorical_cols]
    pool = Pool(X_part, y_part, cat_features=cat_feature_indices)
    return pool, X_part, y_part, categorical_cols


## Обучение ablation-вариантов

Каждый вариант сохраняет промежуточные результаты после обучения, чтобы не потерять прогресс при обрыве Colab-сессии. В режиме `medium` это может занять заметное время, но это всё равно легче, чем запускать полноценный long-run для всех групп признаков.


In [19]:
results = []
topk_rows = []
threshold_rows = []
fitted_models = {}
variant_artifacts = {}

for variant_name, features in feature_variants.items():
    print("\n" + "=" * 100)
    print("Обучаю variant:", variant_name, "features:", len(features))
    start = time.time()

    train_pool, X_train_var, y_train_var, categorical_cols = make_pool(features, train_idx)
    val_pool, X_val_var, y_val_var, _ = make_pool(features, val_idx)
    test_pool, X_test_var, y_test_var, _ = make_pool(features, test_idx)

    model = CatBoostClassifier(**CATBOOST_PARAMS)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True)
    elapsed_sec = time.time() - start

    fitted_models[variant_name] = model
    variant_artifacts[variant_name] = {
        "features": features,
        "categorical_cols": categorical_cols,
        "test_pool": test_pool,
        "val_pool": val_pool,
        "y_test": y_test_var,
        "y_val": y_val_var,
    }

    for split_name, pool, y_part in [
        ("val", val_pool, y_val_var),
        ("test", test_pool, y_test_var),
    ]:
        y_prob = model.predict_proba(pool)[:, 1]
        row = calc_metrics(y_part, y_prob, threshold=0.5)
        row.update({
            "variant": variant_name,
            "split": split_name,
            "n_features": len(features),
            "n_categorical": len(categorical_cols),
            "best_iteration": model.get_best_iteration(),
            "elapsed_sec": elapsed_sec,
        })
        results.append(row)

        topk_df = build_topk_table(y_part, y_prob)
        topk_df["variant"] = variant_name
        topk_df["split"] = split_name
        topk_rows.append(topk_df)

        thresholds_df = build_threshold_table(y_part, y_prob)
        thresholds_df["variant"] = variant_name
        thresholds_df["split"] = split_name
        threshold_rows.append(thresholds_df)

    results_df = pd.DataFrame(results)
    topk_all_df = pd.concat(topk_rows, ignore_index=True)
    thresholds_all_df = pd.concat(threshold_rows, ignore_index=True)

    results_df.to_csv(RESULTS_DIR / "ml_v1_2_ablation_results.csv", index=False)
    topk_all_df.to_csv(RESULTS_DIR / "ml_v1_2_ablation_topk.csv", index=False)
    thresholds_all_df.to_csv(RESULTS_DIR / "ml_v1_2_ablation_thresholds.csv", index=False)

    print("Промежуточные results сохранены в:", RESULTS_DIR)
    display(results_df.sort_values(["split", "pr_auc"], ascending=[True, False]).head(20))

print("Готово. Все ablation results сохранены в:", RESULTS_DIR)



Обучаю variant: full_base features: 82


Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.8385551	test: 0.8329890	best: 0.8329890 (0)	total: 52.9ms	remaining: 1m 14s
200:	learn: 0.9052617	test: 0.8894447	best: 0.8894447 (200)	total: 9.82s	remaining: 58.6s
400:	learn: 0.9197388	test: 0.8965871	best: 0.8966003 (398)	total: 17.1s	remaining: 42.6s
600:	learn: 0.9293789	test: 0.8991490	best: 0.8991490 (600)	total: 27s	remaining: 35.9s
800:	learn: 0.9367004	test: 0.9010306	best: 0.9010540 (798)	total: 35s	remaining: 26.2s
1000:	learn: 0.9426455	test: 0.9021008	best: 0.9021057 (999)	total: 44.4s	remaining: 17.7s
1200:	learn: 0.9475454	test: 0.9023678	best: 0.9026294 (1149)	total: 54.4s	remaining: 9.01s
bestTest = 0.9026294085
bestIteration = 1149
Shrink model to first 1150 iterations.
Промежуточные results сохранены в: /content/drive/MyDrive/ieee-db/ML/1.2/results_ablation


,roc_auc,pr_auc,precision,recall,f1,pred_positive_rate,pred_positive_count,support,positive_support,variant,split,n_features,n_categorical,best_iteration,elapsed_sec
1,0.892295,0.445825,0.208309,0.741273,0.325225,0.132584,3466,26142,974,full_base,test,82,16,1149,59.230253
0,0.893795,0.442336,0.202886,0.736140,0.318101,0.135185,3534,26142,974,full_base,val,82,16,1149,59.230253



Обучаю variant: with_sparse_identity features: 93


...


Обучаю variant: without_flow_risk features: 73


Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.8401662	test: 0.8298775	best: 0.8298775 (0)	total: 55ms	remaining: 1m 16s
200:	learn: 0.9035313	test: 0.8855741	best: 0.8855741 (200)	total: 7.27s	remaining: 43.4s
400:	learn: 0.9180908	test: 0.8937556	best: 0.8937629 (397)	total: 17s	remaining: 42.4s
600:	learn: 0.9277839	test: 0.8969225	best: 0.8969225 (600)	total: 24.4s	remaining: 32.5s
800:	learn: 0.9352129	test: 0.8983401	best: 0.8983784 (799)	total: 34.3s	remaining: 25.7s
1000:	learn: 0.9413819	test: 0.8988938	best: 0.8991135 (942)	total: 44.1s	remaining: 17.6s
bestTest = 0.8991135381
bestIteration = 942
Shrink model to first 943 iterations.
Промежуточные results сохранены в: /content/drive/MyDrive/ieee-db/ML/1.2/results_ablation


,roc_auc,pr_auc,precision,recall,f1,pred_positive_rate,pred_positive_count,support,positive_support,variant,split,n_features,n_categorical,best_iteration,elapsed_sec
1,0.892295,0.445825,0.208309,0.741273,0.325225,0.132584,3466,26142,974,full_base,test,82,16,1149,59.230253
3,0.891695,0.442164,0.200780,0.740246,0.315882,0.137365,3591,26142,974,with_sparse_identity,test,93,18,942,52.029913
13,0.890332,0.439596,0.202489,0.735113,0.317517,0.135261,3536,26142,974,without_flow_risk,test,73,16,942,50.130171
9,0.879049,0.430046,0.189454,0.715606,0.299592,0.140731,3679,26142,974,without_amount,test,71,16,805,42.041709
11,0.877683,0.429694,0.198190,0.697125,0.308636,0.131053,3426,26142,974,without_activity_time,test,77,16,1122,56.588234
7,0.886838,0.424638,0.189974,0.739220,0.302267,0.144977,3790,26142,974,without_email_domain,test,81,15,1020,50.120629
5,0.858321,0.316841,0.143587,0.721766,0.239523,0.187285,4896,26142,974,without_identity,test,28,3,939,30.672814
0,0.893795,0.442336,0.202886,0.736140,0.318101,0.135185,3534,26142,974,full_base,val,82,16,1149,59.230253
12,0.889898,0.437651,0.196317,0.744353,0.310692,0.141267,3693,26142,974,without_flow_risk,val,73,16,942,50.130171
2,0.892463,0.435289,0.196100,0.743326,0.310330,0.141229,3692,26142,974,with_sparse_identity,val,93,18,942,52.029913


Готово. Все ablation results сохранены в: /content/drive/MyDrive/ieee-db/ML/1.2/results_ablation


In [20]:
results_df = pd.DataFrame(results)
topk_all_df = pd.concat(topk_rows, ignore_index=True)
thresholds_all_df = pd.concat(threshold_rows, ignore_index=True)

best_variant = (
    results_df[results_df["split"] == "val"]
    .sort_values("pr_auc", ascending=False)
    .iloc[0]["variant"]
)
print("Лучший variant по val PR-AUC:", best_variant)

display(results_df.sort_values(["split", "pr_auc"], ascending=[True, False]))

test_thresholds = thresholds_all_df[(thresholds_all_df["split"] == "test") & (thresholds_all_df["variant"] == best_variant)]
print("Test thresholds для лучшего варианта:")
display(test_thresholds.sort_values("threshold"))

test_topk = topk_all_df[(topk_all_df["split"] == "test") & (topk_all_df["variant"] == best_variant)]
print("Test top-k для лучшего варианта:")
display(test_topk.sort_values("top_rate"))


Лучший variant по val PR-AUC: full_base


,roc_auc,pr_auc,precision,recall,f1,pred_positive_rate,pred_positive_count,support,positive_support,variant,split,n_features,n_categorical,best_iteration,elapsed_sec
1,0.892295,0.445825,0.208309,0.741273,0.325225,0.132584,3466,26142,974,full_base,test,82,16,1149,59.230253
3,0.891695,0.442164,0.200780,0.740246,0.315882,0.137365,3591,26142,974,with_sparse_identity,test,93,18,942,52.029913
13,0.890332,0.439596,0.202489,0.735113,0.317517,0.135261,3536,26142,974,without_flow_risk,test,73,16,942,50.130171
9,0.879049,0.430046,0.189454,0.715606,0.299592,0.140731,3679,26142,974,without_amount,test,71,16,805,42.041709
11,0.877683,0.429694,0.198190,0.697125,0.308636,0.131053,3426,26142,974,without_activity_time,test,77,16,1122,56.588234
7,0.886838,0.424638,0.189974,0.739220,0.302267,0.144977,3790,26142,974,without_email_domain,test,81,15,1020,50.120629
5,0.858321,0.316841,0.143587,0.721766,0.239523,0.187285,4896,26142,974,without_identity,test,28,3,939,30.672814
0,0.893795,0.442336,0.202886,0.736140,0.318101,0.135185,3534,26142,974,full_base,val,82,16,1149,59.230253
12,0.889898,0.437651,0.196317,0.744353,0.310692,0.141267,3693,26142,974,without_flow_risk,val,73,16,942,50.130171
2,0.892463,0.435289,0.196100,0.743326,0.310330,0.141229,3692,26142,974,with_sparse_identity,val,93,18,942,52.029913


Test thresholds для лучшего варианта:


,roc_auc,pr_auc,precision,recall,f1,pred_positive_rate,pred_positive_count,support,positive_support,threshold,variant,split
6,0.892295,0.445825,0.208309,0.741273,0.325225,0.132584,3466,26142,974,0.50,full_base,test
7,0.892295,0.445825,0.338297,0.583162,0.428194,0.064226,1679,26142,974,0.70,full_base,test
8,0.892295,0.445825,0.387144,0.544148,0.452411,0.052368,1369,26142,974,0.75,full_base,test
9,0.892295,0.445825,0.440662,0.491786,0.464823,0.041581,1087,26142,974,0.80,full_base,test
10,0.892295,0.445825,0.502387,0.432238,0.464680,0.032056,838,26142,974,0.85,full_base,test
11,0.892295,0.445825,0.596522,0.352156,0.442866,0.021995,575,26142,974,0.90,full_base,test


Test top-k для лучшего варианта:


,top_rate,top_k,precision_at_k,recall_at_k,fraud_found,total_fraud,variant,split
5,0.01,262,0.740458,0.199179,194,974,full_base,test
6,0.03,785,0.521019,0.419918,409,974,full_base,test
7,0.05,1308,0.399083,0.535934,522,974,full_base,test
8,0.10,2615,0.252390,0.677618,660,974,full_base,test
9,0.15,3922,0.187659,0.755647,736,974,full_base,test


## Сегментный анализ

Сегменты считаем для лучшего варианта по validation PR-AUC. Основной рабочий порог - `0.80`, плюс сохраняем `0.75` и `0.85` как альтернативные режимы.


In [21]:
best_model = fitted_models[best_variant]
best_artifacts = variant_artifacts[best_variant]
features = best_artifacts["features"]
test_pool = best_artifacts["test_pool"]
y_test_var = best_artifacts["y_test"]
test_prob = best_model.predict_proba(test_pool)[:, 1]

test_meta = df.iloc[test_idx].copy()
segment_defs = {
    "all_test": pd.Series(True, index=test_meta.index),
}

if "low_history_flag" in test_meta.columns:
    segment_defs["low_history_flag=1"] = test_meta["low_history_flag"].fillna(0).astype(float).eq(1)
    segment_defs["low_history_flag=0"] = test_meta["low_history_flag"].fillna(0).astype(float).eq(0)
if "identity_present" in test_meta.columns:
    segment_defs["identity_present=1"] = test_meta["identity_present"].fillna(0).astype(float).eq(1)
    segment_defs["identity_present=0"] = test_meta["identity_present"].fillna(0).astype(float).eq(0)
if "tx_count_total" in test_meta.columns:
    segment_defs["tx_count_total<=2"] = test_meta["tx_count_total"].fillna(0).astype(float).le(2)
    segment_defs["tx_count_total>2"] = test_meta["tx_count_total"].fillna(0).astype(float).gt(2)

segment_rows = []
for segment_name, mask_series in segment_defs.items():
    mask = mask_series.to_numpy()
    y_seg = np.asarray(y_test_var)[mask]
    prob_seg = test_prob[mask]
    if len(y_seg) == 0:
        continue
    for threshold in [0.75, 0.80, 0.85]:
        row = calc_metrics(y_seg, prob_seg, threshold=threshold)
        row.update({
            "variant": best_variant,
            "segment": segment_name,
            "threshold": threshold,
        })
        segment_rows.append(row)

segment_results_df = pd.DataFrame(segment_rows)
segment_results_df.to_csv(RESULTS_DIR / "ml_v1_2_segment_results.csv", index=False)
print("Segment results сохранены в:", RESULTS_DIR / "ml_v1_2_segment_results.csv")
display(segment_results_df.sort_values(["segment", "threshold"]))


Segment results сохранены в: /content/drive/MyDrive/ieee-db/ML/1.2/results_ablation/ml_v1_2_segment_results.csv


,roc_auc,pr_auc,precision,recall,f1,pred_positive_rate,pred_positive_count,support,positive_support,variant,segment,threshold
0,0.892295,0.445825,0.387144,0.544148,0.452411,0.052368,1369,26142,974,full_base,all_test,0.75
1,0.892295,0.445825,0.440662,0.491786,0.464823,0.041581,1087,26142,974,full_base,all_test,0.80
2,0.892295,0.445825,0.502387,0.432238,0.464680,0.032056,838,26142,974,full_base,all_test,0.85
12,0.813967,0.103975,0.207317,0.120996,0.152809,0.009251,164,17728,281,full_base,identity_present=0,0.75
13,0.813967,0.103975,0.178082,0.046263,0.073446,0.004118,73,17728,281,full_base,identity_present=0,0.80
14,0.813967,0.103975,0.218750,0.024911,0.044728,0.001805,32,17728,281,full_base,identity_present=0,0.85
9,0.901052,0.560229,0.411618,0.715729,0.522655,0.143214,1205,8414,693,full_base,identity_present=1,0.75
10,0.901052,0.560229,0.459566,0.672439,0.545987,0.120513,1014,8414,693,full_base,identity_present=1,0.80
11,0.901052,0.560229,0.513648,0.597403,0.552368,0.095793,806,8414,693,full_base,identity_present=1,0.85
6,0.888923,0.553511,0.425764,0.656566,0.516556,0.130447,458,3511,297,full_base,low_history_flag=0,0.75


In [22]:
importance_df = pd.DataFrame({
    "feature": features,
    "importance": best_model.get_feature_importance(),
}).sort_values("importance", ascending=False)

importance_path = RESULTS_DIR / "ml_v1_2_best_feature_importance.csv"
model_path = RESULTS_DIR / "ml_v1_2_best_catboost.cbm"
importance_df.to_csv(importance_path, index=False)
best_model.save_model(str(model_path))

print("Feature importance сохранен в:", importance_path)
print("Best model сохранена в:", model_path)
display(importance_df.head(60))


Feature importance сохранен в: /content/drive/MyDrive/ieee-db/ML/1.2/results_ablation/ml_v1_2_best_feature_importance.csv
Best model сохранена в: /content/drive/MyDrive/ieee-db/ML/1.2/results_ablation/ml_v1_2_best_catboost.cbm


,feature,importance
1,tx_amount_sum,6.863798
27,top_email_domain,6.788427
14,odd_amount_share,6.304850
3,tx_amount_median_proxy,5.405849
2,tx_amount_mean,5.081775
12,short_turnover_share,3.833144
7,tx_sum_stanadart_flow_proxy_proxy,3.436764
11,avg_inter_tx_seconds,3.145359
9,tx_freq_per_day,3.038135
70,id_20_mode,3.012041


## Как читать итог v1.2

Сначала смотрим, какой вариант победил по validation PR-AUC, и подтверждаем его на test. Затем смотрим не только общий PR-AUC, но и top-k: если удаление группы признаков ухудшает top-k, значит эта группа особенно важна для triage. Цель `v1.2` - диагностическая: понять направление. Если найден лучший набор признаков, его стоит потом подтвердить отдельным long-run запуском, а не считать этот быстрый ablation финальной моделью.
